## Scenario 3: Customer Segmentation Query


I am a junior data analyst  . The product team needs to segment customers based on their purchasing behavior for a new feature rollout. The database schema contains: user_activity:
 user_id
last_login_date
feature_usage_count
account_type
transactions:
 transaction_id
user_id
transaction_date
amount
platform
user_preferences:
 user_id
communication_preference
interface_theme
notification_settings. Create a sql query to identify ; Active users (logged in last 30 days), Filter by high-value customers (top 20% by spending), User preference trends for the identified customers

In [2]:
# Run this cell to execute the query against a SQLite in-memory database
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ── Build demo database ───────────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')

np.random.seed(0)
n_users = 500

users_df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'last_login_date': [
        (datetime.today() - timedelta(days=int(d))).strftime('%Y-%m-%d')
        for d in np.random.randint(0, 60, n_users)
    ],
    'feature_usage_count': np.random.randint(0, 200, n_users),
    'account_type': np.random.choice(
        ['free', 'basic', 'premium'], n_users
    )
})

txn_df = pd.DataFrame({
    'transaction_id': range(1, 2001),
    'user_id': np.random.randint(1, n_users + 1, 2000),
    'transaction_date': [
        (datetime.today() - timedelta(days=int(d))).strftime('%Y-%m-%d')
        for d in np.random.randint(0, 90, 2000)
    ],
    'amount': np.round(np.random.exponential(50, 2000), 2),
    'platform': np.random.choice(
        ['web', 'ios', 'android'], 2000
    )
})

prefs_df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'communication_preference': np.random.choice(
        ['email', 'sms', 'push'], n_users
    ),
    'interface_theme': np.random.choice(
        ['light', 'dark', 'system'], n_users
    ),
    'notification_settings': np.random.choice(
        ['all', 'important', 'none'], n_users
    )
})


users_df.to_sql(
    'user_activity',
    conn,
    if_exists='replace',
    index=False
)

txn_df.to_sql(
    'transactions',
    conn,
    if_exists='replace',
    index=False
)

prefs_df.to_sql(
    'user_preferences',
    conn,
    if_exists='replace',
    index=False
)


# ── Customer segmentation query ──────────────────────────────────────────────

query = """

WITH active_users AS (

    -- Users logged in within last 30 days
    SELECT
        user_id,
        account_type,
        feature_usage_count,
        CAST(
            julianday('now') - julianday(last_login_date)
            AS INTEGER
        ) AS days_since_login

    FROM user_activity

    WHERE last_login_date >= date('now', '-30 days')
),


user_spending AS (

    -- Total spending per user
    SELECT
        user_id,
        SUM(amount) AS total_spend

    FROM transactions

    GROUP BY user_id
),


ranked_users AS (

    -- Rank users by spending
    SELECT
        user_id,
        total_spend,

        NTILE(5) OVER (
            ORDER BY total_spend DESC
        ) AS spend_quintile

    FROM user_spending
),


high_value_active AS (

    -- Active + top 20% spenders
    SELECT
        au.user_id,
        au.account_type,
        au.feature_usage_count,
        au.days_since_login,

        ru.total_spend,
        ru.spend_quintile

    FROM active_users au

    INNER JOIN ranked_users ru
        ON au.user_id = ru.user_id

    WHERE ru.spend_quintile = 1
),


preferred_platform AS (

    -- Most used platform per customer
    SELECT
        user_id,
        platform,

        ROW_NUMBER() OVER(
            PARTITION BY user_id
            ORDER BY COUNT(*) DESC
        ) AS rn

    FROM transactions

    GROUP BY user_id, platform
)


SELECT

    hva.user_id,
    hva.account_type,

    ROUND(hva.total_spend, 2)
        AS total_spend,

    hva.feature_usage_count,
    hva.days_since_login,

    pp.platform
        AS most_used_platform,

    up.communication_preference,
    up.interface_theme,
    up.notification_settings,

    hva.spend_quintile
        AS spend_bucket


FROM high_value_active hva


LEFT JOIN preferred_platform pp
    ON hva.user_id = pp.user_id
    AND pp.rn = 1


LEFT JOIN user_preferences up
    ON hva.user_id = up.user_id


ORDER BY total_spend DESC;

"""


result = pd.read_sql_query(query, conn)


print(
    f"High-value active customers found: {len(result)}"
)

print(
    result.head(10).to_string(index=False)
)


conn.close()

High-value active customers found: 54
 user_id account_type  total_spend  feature_usage_count  days_since_login most_used_platform communication_preference interface_theme notification_settings  spend_bucket
     358         free      1043.94                   12                20                web                    email          system                  none             1
     437      premium       659.76                   47                 4                web                     push          system             important             1
     465        basic       641.79                   64                18            android                      sms          system                  none             1
     373         free       605.54                  153                30            android                     push           light                  none             1
     182      premium       518.87                  166                 0                web                    